<h1 style="color:rgb(199, 12, 77);">Data Cleaning and Feature Engineering</h1>

The purpose of this notebook is to clean the Bitcoin historical dataset and create meaningful features for regression modeling. These features will help the machine learning models better understand price trends, volatility, momentum, and time-based market behavior.

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
df = pd.read_csv(
    "../data/processed/bitcoin_processed.csv",
    parse_dates=["Date"]
)

In [14]:
df.iloc[3120000:3120005]

,Timestamp,Open,High,Low,Close,Volume,Date,Volatility,Year,Price_change,Return,MA_60,MA_240,MA_1440,Rolling_Volatility,Month,Hour,Range,Cumulative_Return
3120000,1.512612e+09,13180.04,13195.00,13180.04,13194.98,1.783720,2017-12-07 02:01:00,14.96,2017,14.94,0.001134,13112.136667,12884.950125,12380.596569,0.003059,December,2,14.96,5.141451e-13
3120001,1.512612e+09,13194.98,13194.99,13173.00,13190.00,10.862599,2017-12-07 02:02:00,21.99,2017,-4.98,-0.000377,13114.386667,12887.054333,12381.596569,0.002952,December,2,21.99,5.139511e-13
3120002,1.512612e+09,13173.01,13190.00,13173.01,13176.91,4.799181,2017-12-07 02:03:00,16.99,2017,3.90,0.000296,13118.061167,12889.125333,12382.593785,0.002769,December,2,16.99,5.141032e-13
3120003,1.512612e+09,13189.33,13189.34,13160.00,13175.11,6.295512,2017-12-07 02:04:00,29.34,2017,-14.22,-0.001078,13120.229333,12891.302750,12383.589750,0.002642,December,2,29.34,5.135489e-13
3120004,1.512612e+09,13189.32,13189.33,13157.05,13175.00,5.933937,2017-12-07 02:05:00,32.28,2017,-14.32,-0.001086,13122.445500,12893.319375,12384.581271,0.002647,December,2,32.28,5.129914e-13


In [31]:
df = df.copy()

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7535212 entries, 1439 to 7536650
Data columns (total 21 columns):
 #   Column              Dtype         
---  ------              -----         
 0   Timestamp           float64       
 1   Open                float64       
 2   High                float64       
 3   Low                 float64       
 4   Close               float64       
 5   Volume              float64       
 6   Date                datetime64[ns]
 7   Volatility          float64       
 8   Year                int64         
 9   Price_change        float64       
 10  Return              float64       
 11  MA_60               float64       
 12  MA_240              float64       
 13  MA_1440             float64       
 14  Rolling_Volatility  float64       
 15  Month               int64         
 16  Hour                int64         
 17  Range               float64       
 18  Cumulative_Return   float64       
 19  DayOfWeek           int32         
 20  Next

In [33]:
df.isnull().sum()

Timestamp             0
Open                  0
High                  0
Low                   0
Close                 0
Volume                0
Date                  0
Volatility            0
Year                  0
Price_change          0
Return                0
MA_60                 0
MA_240                0
MA_1440               0
Rolling_Volatility    0
Month                 0
Hour                  0
Range                 0
Cumulative_Return     0
DayOfWeek             0
Next_Close            0
dtype: int64

In [34]:
df.shape

(7535212, 21)

In [35]:
df = df.dropna()

In [36]:
df.reset_index()

,index,Timestamp,Open,High,Low,Close,Volume,Date,Volatility,Year,...,MA_60,MA_240,MA_1440,Rolling_Volatility,Month,Hour,Range,Cumulative_Return,DayOfWeek,Next_Close
0,1439,1.325498e+09,5.0,5.0,5.0,5.0,0.000000,2012-01-02 10:00:00,0.0,2012,...,5.000000,5.000000,4.768569,0.000000,1,10,0.0,1.000000e+00,0,5.0
1,1440,1.325498e+09,5.0,5.0,5.0,5.0,0.000000,2012-01-02 10:01:00,0.0,2012,...,5.000000,5.000000,4.768861,0.000000,1,10,0.0,1.000000e+00,0,5.0
2,1441,1.325499e+09,5.0,5.0,5.0,5.0,0.000000,2012-01-02 10:02:00,0.0,2012,...,5.000000,5.000000,4.769153,0.000000,1,10,0.0,1.000000e+00,0,5.0
3,1442,1.325499e+09,5.0,5.0,5.0,5.0,0.000000,2012-01-02 10:03:00,0.0,2012,...,5.000000,5.000000,4.769444,0.000000,1,10,0.0,1.000000e+00,0,5.0
4,1443,1.325499e+09,5.0,5.0,5.0,5.0,0.000000,2012-01-02 10:04:00,0.0,2012,...,5.000000,5.000000,4.769736,0.000000,1,10,0.0,1.000000e+00,0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7535207,7536646,1.777680e+09,78251.0,78251.0,78251.0,78251.0,0.000317,2026-05-02 00:07:00,0.0,2026,...,78137.566667,78139.154167,77647.237500,0.000188,5,0,0.0,5.668671e-14,5,78245.0
7535208,7536647,1.777680e+09,78250.0,78250.0,78244.0,78245.0,0.233410,2026-05-02 00:08:00,6.0,2026,...,78138.983333,78138.854167,77648.502083,0.000188,5,0,6.0,5.668308e-14,5,78254.0
7535209,7536648,1.777681e+09,78245.0,78254.0,78238.0,78254.0,0.792507,2026-05-02 00:09:00,16.0,2026,...,78141.183333,78138.587500,77649.738194,0.000178,5,0,16.0,5.668960e-14,5,78256.0
7535210,7536649,1.777681e+09,78257.0,78257.0,78250.0,78256.0,0.101332,2026-05-02 00:10:00,7.0,2026,...,78142.750000,78138.375000,77650.988194,0.000166,5,0,7.0,5.668888e-14,5,78251.0


In [37]:
df.isnull().sum()

Timestamp             0
Open                  0
High                  0
Low                   0
Close                 0
Volume                0
Date                  0
Volatility            0
Year                  0
Price_change          0
Return                0
MA_60                 0
MA_240                0
MA_1440               0
Rolling_Volatility    0
Month                 0
Hour                  0
Range                 0
Cumulative_Return     0
DayOfWeek             0
Next_Close            0
dtype: int64

In [38]:
df.duplicated().sum()

np.int64(0)

In [39]:
df["DayOfWeek"] = df["Date"].dt.dayofweek

In [40]:
df["Month"] = df["Date"].dt.month.astype(int)

In [41]:
df["Next_Close"] = df["Close"].shift(-1)

In [42]:
df = df.dropna()

In [48]:
df.columns

Index(['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume', 'Date',
       'Volatility', 'Year', 'Price_change', 'Return', 'MA_60', 'MA_240',
       'MA_1440', 'Rolling_Volatility', 'Month', 'Hour', 'Range',
       'Cumulative_Return', 'DayOfWeek', 'Next_Close'],
      dtype='object')

In [50]:
df = df.drop(["Timestamp", "Date", "Range", "Cumulative_Return"], axis=1)

In [51]:
df["Log_Return"] = np.log(df["Close"] / df["Open"])

In [52]:
df["Close_Lag_1"] = df["Close"].shift(1)
df["Close_Lag_5"] = df["Close"].shift(5)
df["Close_Lag_15"] = df["Close"].shift(15)

df["Return_Lag_1"] = df["Return"].shift(1)
df["Volume_Lag_1"] = df["Volume"].shift(1)

In [53]:
df = df.dropna()

### Final Feature Set

Target:
Next_Close

Features:
Open, High, Low, Close, Volume  
Volatility, Price_change, Return, Log_Return  
MA_60, MA_240, MA_1440  
Rolling_Volatility  
Year, Month, Hour, DayOfWeek  
Close_Lag_1, Close_Lag_5, Close_Lag_15  
Return_Lag_1, Volume_Lag_1

In [57]:
df = df.reset_index(drop=True)

In [58]:
df.drop(columns=["Price_change"], inplace=True)

In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7535196 entries, 0 to 7535195
Data columns (total 22 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Open                float64
 1   High                float64
 2   Low                 float64
 3   Close               float64
 4   Volume              float64
 5   Volatility          float64
 6   Year                int64  
 7   Return              float64
 8   MA_60               float64
 9   MA_240              float64
 10  MA_1440             float64
 11  Rolling_Volatility  float64
 12  Month               int64  
 13  Hour                int64  
 14  DayOfWeek           int32  
 15  Next_Close          float64
 16  Log_Return          float64
 17  Close_Lag_1         float64
 18  Close_Lag_5         float64
 19  Close_Lag_15        float64
 20  Return_Lag_1        float64
 21  Volume_Lag_1        float64
dtypes: float64(18), int32(1), int64(3)
memory usage: 1.2 GB


In [59]:
df.to_csv("../data/processed/bitcoin_model_ready.csv", index=False)

<h1 style="color:rgb(199, 12, 77);">Feature Engineering Summary</h1>

In this notebook, the dataset was cleaned and transformed into a model-ready format.

Time-based features, price-based features, return-based features, lag features, and rolling statistics were created.

The target variable was defined as Next_Close.

The dataset is now ready for machine learning modeling.